# 02 - Data Cleaning

This notebook walks through the data cleaning process using our reusable ETL pipeline script.

In [1]:
import pandas as pd
import sys
import os

# Add scripts folder to sys.path
sys.path.append('../')

from scripts.etl_pipeline import AmazonDataCleaner

### Initialize Cleaner

In [2]:
RAW_DATA = '../data/raw/amazon_products_sales_data_uncleaned.csv'
CLEANED_DATA = '../data/processed/cleaned_data.csv'

cleaner = AmazonDataCleaner(RAW_DATA, CLEANED_DATA)

### Run Pipeline Step-by-Step
We can run individual methods for an interactive view, or run the entire pipeline at once.

In [3]:
cleaner.load_data()
print(f'Initial Shape: {cleaner.df.shape}')

2026-04-22 12:08:46,290 - INFO - Loading raw data from ../data/raw/amazon_products_sales_data_uncleaned.csv...


2026-04-22 12:08:46,558 - INFO - Loaded 42675 rows and 16 columns.


Initial Shape: (42675, 16)


In [4]:
cleaner.remove_duplicates()
cleaner.clean_rating()
cleaner.clean_reviews()
cleaner.clean_bought_last_month()
cleaner.clean_prices()
cleaner.engineer_features()

# NOTE: assign_category() is called in the next dedicated cell (Step 4).
# Guardrails (price filter) are applied AFTER category assignment to ensure
# every row gets a category before any rows are dropped.


2026-04-22 12:08:46,610 - INFO - Removed 0 duplicate rows.


2026-04-22 12:08:46,610 - INFO - Cleaning 'rating' column...


2026-04-22 12:08:46,645 - INFO - Cleaning 'number_of_reviews' column...


2026-04-22 12:08:46,659 - INFO - Cleaning 'bought_in_last_month' column...


2026-04-22 12:08:46,674 - INFO - Cleaning price columns...


2026-04-22 12:08:46,707 - INFO - Engineering boolean flags and standardizing formats...


---
## Step 4: Category Derivation

**Why this step?**  
The problem statement requires *category-level* analysis — discount depth, rating distribution,
and review engagement broken down by product type. Without a `product_category` column, all
segment-level insights are impossible.

**Why keyword-based inference?**  
The raw dataset has **no explicit category column**. All rows were scraped from Amazon's
`s=electronics` search department. The `title` field is the richest structured signal available
and maps reliably to subcategories (e.g., *'MacBook Air'* → `Laptops & Computers`).

**Three modular functions are used:**
- `extract_category(title)` — keyword rule-based classification
- `clean_category(series)` — standardises casing/whitespace
- `assign_category()` — orchestrates derivation + validation

These are defined in `scripts/etl_pipeline.py` and callable directly on the cleaner instance.

In [ ]:
# ── extract_category: keyword lookup (already defined in AmazonDataCleaner) ─
# Shown here for transparency — this is the exact rule dictionary used.
# Priority order: more specific categories appear BEFORE broad ones.
# Example: 'Microphones' checked before 'Audio & Speakers' to avoid false matches.

print('Category rules loaded from AmazonDataCleaner.CATEGORY_RULES:')
for cat, kws in cleaner.CATEGORY_RULES.items():
    print(f'  {cat}: {kws[:3]}...')


In [ ]:
# ── assign_category: apply + validate ───────────────────────────────────────
# Calls extract_category() on every title, then clean_category() to standardise.
# Built-in assertion: will raise AssertionError if any null sneaks through.

cleaner.assign_category()


In [ ]:
# ── Validation Checks ────────────────────────────────────────────────────────
# These three checks confirm the column is analysis-ready before export.

df = cleaner.df  # alias for readability

# CHECK 1: Null count — must be 0
null_count = df['product_category'].isna().sum()
print(f'Null count in product_category: {null_count}')
assert null_count == 0, 'FAIL: Null values found in product_category!'
print('PASS: Zero nulls in product_category')
print()

# CHECK 2: Unique categories
unique_cats = df['product_category'].unique()
print(f'Unique categories ({len(unique_cats)} total):')
for c in sorted(unique_cats):
    print(f'  - {c}')
print()

# CHECK 3: Distribution — confirms no category is disproportionately dominant
print('Category distribution (count & %):')
dist = df['product_category'].value_counts()
pct  = (dist / len(df) * 100).round(1)
print(pd.concat([dist, pct], axis=1, keys=['count', '%']).to_string())
print()

# CHECK 4: Row count unchanged
print(f'Row count after assign_category(): {len(df)}')
print(f'Column count: {df.shape[1]}  (expected 16 — 15 original + product_category)')


In [ ]:
# ── Outlier / Null-Price Filter ─────────────────────────────────────────────
# Drop rows where current_price is NaN (no price ever scraped) OR out of
# valid range (≤ 0 or > 50,000).
# WHY here: clean_prices() uses cross-column imputation; only rows with BOTH
# prices unparseable survive as NaN to this point.

before = len(cleaner.df)
cleaner.df = cleaner.df[
    cleaner.df['current_price'].notna() &
    (cleaner.df['current_price'] > 0) &
    (cleaner.df['current_price'] < 50000)
]
after = len(cleaner.df)
print(f'Rows before price filter: {before}')
print(f'Rows after  price filter: {after}')
print(f'Rows dropped (no/invalid price): {before - after}')


---
## Step 5: Null Value Imputation

**Why this step?**  
After all structural cleaning steps, three text columns still contain null values
that the prior steps could not resolve. Leaving them as `NaN` would silently break
groupby aggregations, Tableau filters, and downstream analysis.

**Strategy: semantic sentinel strings (not statistical imputation)**  
These are categorical/text columns — using mean/median is meaningless. Instead,
each null is replaced with a string that honestly describes *why* the value is missing:

| Column | Null Count | Fill Value | Reason |
|--------|-----------|-----------|--------|
| `buy_box_availability` | 3,046 (9.85%) | `'Unavailable'` | No active Buy Box during scraping (out of stock / no Buy Box winner) |
| `delivery_details` | 113 (0.37%) | `'Not Available'` | Renewed / digital / Protection Plan listings had no delivery estimate |
| `product_url` | 1,789 (5.78%) | `'Unknown'` | SSPA/JS-rendered URLs not captured; rows retain full analytical value |


In [ ]:
# ── fix_nulls: impute remaining text-column nulls ───────────────────────────
# Fills buy_box_availability, delivery_details, and product_url with semantic
# sentinel strings. Includes a scoped assertion to confirm zero text-col nulls.

cleaner.fix_nulls()


In [ ]:
# ── Null Validation ─────────────────────────────────────────────────────────
# Confirm every column in the final dataframe is null-free before export.

df = cleaner.df

total_nulls = df.isna().sum().sum()
print(f'Total nulls across all columns: {total_nulls}')
print()

print('Per-column null count:')
null_report = df.isna().sum().reset_index()
null_report.columns = ['column', 'null_count']
null_report['status'] = null_report['null_count'].apply(lambda x: '✓' if x == 0 else f'✗ ({x})')
print(null_report[['column', 'null_count', 'status']].to_string(index=False))
print()

# CHECK: buy_box_availability filled values
print('buy_box_availability value distribution after fill:')
print(df['buy_box_availability'].value_counts().to_string())
print()

# CHECK: delivery_details — spot 'Not Available'
not_avail = (df['delivery_details'] == 'Not Available').sum()
print(f'delivery_details rows filled as Not Available: {not_avail}')
print()

# CHECK: product_url — spot 'Unknown'
unknown_urls = (df['product_url'] == 'Unknown').sum()
print(f'product_url rows filled as Unknown: {unknown_urls}')
print()

assert total_nulls == 0, f'FAIL: {total_nulls} nulls remain!'
print('ALL NULL CHECKS PASSED ✓')


### Save Cleaned Data

In [5]:
cleaner.save_data()
print(f'Final Shape: {cleaner.df.shape}')

cleaner.df.head()

2026-04-22 12:08:47,019 - INFO - Cleaned data saved to ../data/processed/cleaned_data.csv. Final shape: (30926, 15)


Final Shape: (30926, 15)


,title,rating,number_of_reviews,bought_in_last_month,listed_price,is_best_seller,is_sponsored,buy_box_availability,delivery_details,image_url,product_url,collected_at,current_price,has_coupon,is_sustainable
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375,300,159.00,0,1,Add to cart,"Delivery Mon, Sep 1",https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29,89.68,1,1
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457,6000,15.99,0,1,Add to cart,"Delivery Fri, Aug 29",https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29,9.99,0,0
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044,2000,349.00,0,1,Add to cart,"Delivery Mon, Sep 1",https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29,314.00,0,0
8,Complete Protect: One plan covers all eligible...,4.0,4380,0,16.99,0,0,NaN,NaN,https://m.media-amazon.com/images/I/71tZ0n2xP9...,/Asurion-Complete-Protect/dp/B07RZ3LSHM/ref=sr...,2025-08-21 11:14:29,16.99,1,0
10,Amazon Basics 48-Pack AA Alkaline High-Perform...,4.7,865598,100000,14.99,1,0,Add to cart,"Delivery Fri, Aug 29",https://m.media-amazon.com/images/I/81iJ+tnLAD...,/AmazonBasics-Performance-Alkaline-Batteries-C...,2025-08-21 11:14:29,14.99,0,1
